In [0]:
data = [
    (1,'John',"New York")
]
initial_df = spark.createDataFrame(data,["id","name","address"])

In [0]:
from pyspark.sql.functions import lit,to_date

In [0]:
target_df = initial_df.withColumn('current_date',to_date(lit('2025-06-05'))) \
    .withColumn('is_current',lit('Y')) \
        .withColumn("end_date",lit(None).cast("date"))


In [0]:
target_df.write.mode("overwrite").format("delta").saveAsTable("employee_scd2")

In [0]:
data = [
    (1,"John","Washington Dc")
]
changed_df = spark.createDataFrame(data,["id","name","address"]) \
    .withColumn("changed_date",to_date(lit("2025-06-06")))

In [0]:
changed_df.write.mode("overwrite").format("delta").saveAsTable("employee_changes")

### 1. Expire old current record
### 2. Insert new current record

In [0]:
spark.sql("""
          MERGE INTO employee_scd2 t 
          USING employee_changes s 
          ON t.id = s.id AND
          t.is_current = 'Y' AND 
          t.address <> s.address
          WHEN MATCHED THEN
          UPDATE SET t.is_current = 'N',
          t.end_date = s.changed_date,
          t.address = s.address
          
          """)

In [0]:
spark.sql("select * from employee_scd2").show()

In [0]:
from pyspark.sql.functions import to_date, lit

In [0]:
data = [
    (1, "John", "New York")
]

columns = ["emp_id", "name", "address"]

df = spark.createDataFrame(data, columns)

target_df = df \
    .withColumn("effective_date", to_date(lit("2025-06-01"))) \
    .withColumn("end_date", lit(None).cast("date")) \
    .withColumn("is_current", lit("Y"))

target_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("employee_scd21")

spark.table("employee_scd21").show()

In [0]:
change_data = [
    (1, "John", "Chicago", "2025-06-02")
]

change_columns = ["emp_id", "name", "address", "change_date"]

source_df = spark.createDataFrame(change_data, change_columns) \
    .withColumn("change_date", to_date("change_date"))

source_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("employee_source_change")

spark.table("employee_source_change").show()

In [0]:
spark.sql("""
MERGE INTO employee_scd21 t
USING employee_source_change s
ON t.emp_id = s.emp_id
AND t.is_current = 'Y'
AND t.address <> s.address

WHEN MATCHED THEN
  UPDATE SET
    t.end_date = s.change_date,
    t.is_current = 'N'
""")



In [0]:
spark.table("employee_scd21").show()

In [0]:
new_version_df = spark.sql("""
SELECT
  s.emp_id,
  s.name,
  s.address,
  s.change_date AS effective_date,
  CAST(NULL AS DATE) AS end_date,
  'Y' AS is_current
FROM employee_source_change s
LEFT JOIN employee_scd21 t
  ON s.emp_id = t.emp_id
 AND t.is_current = 'Y'
WHERE t.emp_id IS NULL
""")

new_version_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("employee_scd21")